# Apresentação dos Resultados - Ideathon Ceres 2026

Este notebook apresenta o relatório estatístico, gráfico e analítico consolidado do **Ideathon de Inovação em Saúde de Ceres (GO)**, realizado em outubro de 2026.

### Sumário
1. Consolidado Geral de Avaliação por Grupo
2. Correlação e Relação entre o Perfil do Participante e Avaliações
3. Modelo de Regressão Linear para Previsão de Interesse Futuro
4. Dispersão Detalhada: Continuidade vs. Interesse


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm

# Carregar dados tratados do Ideathon
excel_path = "/content/sample_data/ideathon_2026/Avaliacao_Ideathon_Cers_Out_26.xlsx"
df = pd.read_excel(excel_path, sheet_name='Avaliaacao_Ideathon_ Out_26')
print("Dados carregados com sucesso! Formato:", df.shape)

## 1. Média Geral das Avaliações por Grupo de Atuação

Compara as avaliações médias (Escala Likert de 1 a 5) entre Participantes, Demandantes e Moderadores.

In [ ]:
# Filtro e cálculo de médias gerais
p_cols = [
    'Como você avalia a experiência de propor soluções voltadas para a saúde de Ceres (GO)? [Aprendizado sobre os problemas da saúde local]',
    'Como você avalia a experiência de propor soluções voltadas para a saúde de Ceres (GO)? [Integração com a equipe]',
    'Como você avalia a experiência de propor soluções voltadas para a saúde de Ceres (GO)? [Desafio de criar uma solução em curto tempo]',
    'Qual a probabilidade de você continuar desenvolvendo a solução após o evento?',
    'O Ideathon aumentou seu interesse em participar de projetos de inovação social/saúde na região de Ceres?'
]
media_part = df[df['Qual é o seu perfil de atuação no evento?'].str.contains('Participante', na=False)][p_cols].mean().mean()

d_cols = [
    'As soluções apresentadas atenderam às demandas reais e prioritárias da rede de saúde local?',
    'Avalie o potencial de adoção das propostas pela gestão pública ou instituições locais: [Viabilidade financeira de implementação]',
    'Avalie o potencial de adoção das propostas pela gestão pública ou instituições locais: [Alinhamento com as normas legais do SUS]',
    'Avalie o potencial de adoção das propostas pela gestão pública ou instituições locais: [Facilidade de adesão pelas equipes de saúde atuais]'
]
media_dem = df[df['Qual é o seu perfil de atuação no evento?'].str.contains('Demandante', na=False)][d_cols].mean().mean()

m_cols = [
    'Em relação ao desempenho das equipes que você acompanhou, como você avalia os pontos abaixo? [Compreensão dos gargalos da saúde em Ceres]',
    'Em relação ao desempenho das equipes que você acompanhou, como você avalia os pontos abaixo? [Capacidade de resposta aos feedbacks]',
    'Em relação ao desempenho das equipes que você acompanhou, como você avalia os pontos abaixo? [Evolução do projeto do início ao fim]',
    'As soluções propostas apresentaram fundamentação coerente com a realidade do SUS em Ceres?',
    'Qual é o nível de maturidade médio dos projetos apresentados para avançarem para uma fase de incubação/piloto?'
]
media_mod = df[df['Qual é o seu perfil de atuação no evento?'].str.contains('Moderador', na=False)][m_cols].mean().mean()

dados_grafico = pd.DataFrame({
    'Grupo': ['Participantes
(N=34)', 'Demandantes
(N=2)', 'Moderadores
(N=2)'],
    'Média': [media_part, media_dem, media_mod]
})

plt.figure(figsize=(8, 5))
sns.set_theme(style="whitegrid")
ax = sns.barplot(x='Grupo', y='Média', data=dados_grafico, palette='viridis', hue='Grupo', legend=False)
for p in ax.patches:
    ax.annotate(f"{p.get_height():.2f}", (p.get_x() + p.get_width() / 2., p.get_height() + 0.05),
                ha='center', va='center', xytext=(0, 5), textcoords='offset points', fontweight='bold')
plt.title('Comparativo Geral de Avaliações por Grupo', fontsize=12, fontweight='bold')
plt.ylim(0, 5.5)
plt.show()

## 2. Relação: Vínculo do Participante vs. Avaliações (Heatmap)

Permite compreender as nuances e as opiniões das subcategorias de participantes ativos.

In [ ]:
part_df = df[df['Qual é o seu perfil de atuação no evento?'].str.contains('Participante', na=False)].copy()
mapping_cols = {
    'Como você avalia a experiência de propor soluções voltadas para a saúde de Ceres (GO)? [Aprendizado sobre os problemas da saúde local]': 'Aprendizado Local',
    'Como você avalia a experiência de propor soluções voltadas para a saúde de Ceres (GO)? [Integração com a equipe]': 'Integração Equipe',
    'Como você avalia a experiência de propor soluções voltadas para a saúde de Ceres (GO)? [Desafio de criar uma solução em curto tempo]': 'Desafio Tempo',
    'Qual a probabilidade de você continuar desenvolvendo a solução após o evento?': 'Continuidade',
    'O Ideathon aumentou seu interesse em participar de projetos de inovação social/saúde na região de Ceres?': 'Aumento Interesse'
}
part_df_mapped = part_df.rename(columns=mapping_cols)
analise_perfil = part_df_mapped.groupby('Qual é o seu vínculo principal no evento?')[list(mapping_cols.values())].mean()

plt.figure(figsize=(10, 5))
sns.heatmap(analise_perfil, annot=True, cmap='YlGnBu', fmt=".2f", linewidths=.5)
plt.title('Vínculo do Participante vs. Avaliações', fontsize=12, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 3. Modelo de Regressão Linear (OLS)

Previsão do interesse futuro baseado nas outras experiências do evento.

In [ ]:
mapping_reg = {
    'Como você avalia a experiência de propor soluções voltadas para a saúde de Ceres (GO)? [Aprendizado sobre os problemas da saúde local]': 'Aprendizado_Local',
    'Como você avalia a experiência de propor soluções voltadas para a saúde de Ceres (GO)? [Integração com a equipe]': 'Integracao_Equipe',
    'Como você avalia a experiência de propor soluções voltadas para a saúde de Ceres (GO)? [Desafio de criar uma solução em curto tempo]': 'Desafio_Tempo',
    'Qual a probabilidade de você continuar desenvolvendo a solução após o evento?': 'Continuidade',
    'O Ideathon aumentou seu interesse em participar de projetos de inovação social/saúde na região de Ceres?': 'Aumento_Interesse'
}
reg_df = part_df.rename(columns=mapping_reg)[list(mapping_reg.values())].dropna()
X = reg_df[['Aprendizado_Local', 'Integracao_Equipe', 'Desafio_Tempo', 'Continuidade']]
y = reg_df['Aumento_Interesse']

X_const = sm.add_constant(X)
model_sm = sm.OLS(y, X_const).fit()
print(model_sm.summary())

## 4. Dispersão Detalhada com Linha de Tendência (Jitter)

Visualização focada no preditor com significância estatística (`Continuidade` vs. `Aumento de Interesse`).

In [ ]:
np.random.seed(42)
jitter_x = reg_df['Continuidade'] + np.random.uniform(-0.15, 0.15, len(reg_df))
jitter_y = reg_df['Aumento_Interesse'] + np.random.uniform(-0.15, 0.15, len(reg_df))

plt.figure(figsize=(8, 5.5))
sns.regplot(x=reg_df['Continuidade'], y=reg_df['Aumento_Interesse'], scatter=False, color='#e74c3c', line_kws={'ls': '--', 'lw': 2})
plt.scatter(jitter_x, jitter_y, color='#2c3e50', alpha=0.7, edgecolors='black', s=80)
plt.title('Probabilidade de Continuidade vs. Aumento de Interesse', fontsize=12, fontweight='bold')
plt.xlabel('Continuidade (1-5)')
plt.ylabel('Aumento de Interesse (1-5)')
plt.xlim(0.5, 5.5)
plt.ylim(0.5, 5.5)
plt.show()